# Data Migration SQL Server to Postgres

In [3]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

## 1. Load Credential environment variables

In [4]:
load_dotenv()  # Load environment variables from .env file

True

In [5]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_database = os.getenv("SQL_SERVER_DB")
sql_user = os.getenv("SQL_SERVER_USER")

In [6]:
print(f"SQL Server Host: {sql_host}")
print(f"SQL Server Database: {sql_database}")
print(f"SQL Server User: {sql_user}")

SQL Server Host: (local)\SQLEXPRESS
SQL Server Database: transactionDB_UAT
SQL Server User: windows


In [7]:
pg_host=os.getenv("PROGRES_HOST")
pg_port=os.getenv("PROGRES_PORT")
pg_database=os.getenv("PROGRES_DB")
pg_user=os.getenv("PROGRES_USER")
pg_password=os.getenv("PROGRES_PASSWORD") 

In [8]:
print (f"PostgreSQL Host: {pg_host}")
print (f"PostgreSQL Port: {pg_port}")
print (f"PostgreSQL Database: {pg_database}")
print (f"PostgreSQL User: {pg_user}")
print (f"PostgreSQL Password: {pg_password}")

PostgreSQL Host: localhost
PostgreSQL Port: 5432
PostgreSQL Database: transaction_uat
PostgreSQL User: postgres
PostgreSQL Password: 123456


## 2. Connect to SQL Server

In [59]:
print("Connecting to SQL Server...")
print(f"    Server: {sql_host}")
print(f"    Database: {sql_database}")

Connecting to SQL Server...
    Server: (local)\SQLEXPRESS
    Database: transactionDB_UAT


In [9]:
try:
    sql_conn_string = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_database};"
        f"Trusted_Connection=yes;"
    )

    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print("[SUCCESS] Connected to SQL Server")
except Exception as e:
    print(f"Error connecting to SQL Server: {e}")
    print(""" How to troubleshoot:
        >1. Check server name in .env file is correct
        >2. Verify that the SQL Server is running
        >3. Ensure that the ODBC Driver 17 for SQL Server is installed
        >4. Check if you can connect to the SQL Server using SQL Server Management Studio (SSMS) or another database client
        >5. If using a remote SQL Server, ensure that the server allows remote connections and that your IP address is whitelisted
        >6. If using Windows Authentication, ensure that your Windows user has the necessary permissions to access the SQL Server database
        >7. If using SQL Server Authentication, ensure that the username and password in the .env file are correct
        >8. Check if there are any firewall rules blocking the connection to the SQL Server
        >9. If the SQL Server is running on a non-default port, make sure to specify the correct port in the .env file
          """)

[SUCCESS] Connected to SQL Server


## 3. Connect to PostgresSQL

In [10]:
print("Connecting to PostgreSQL...")
print(f"    Server: {pg_host}")
print(f"    Database: {pg_database}")

Connecting to PostgreSQL...
    Server: localhost
    Database: transaction_uat


In [11]:
try:
    pg_conn = psycopg2.connect(
        host = pg_host,
        port = pg_port,
        database = pg_database,
        user = pg_user,
        password = pg_password
)
    pg_cursor = pg_conn.cursor()
    pg_cursor.execute("SELECT version();")
    pg_version = pg_cursor.fetchone()[0]

    print(f"[SUCCESS] Connected to PostgreSQL: {pg_version[:50]}...\n")

except psycopg2.OperationalError as e:
    print(f"[ERROR] Failed to connect to PostgreSQL: {e}")
    print(""" How to troubleshoot:
        >1. Check if PostgreSQL server is running and accessible
        >2. Check server name, port, database name, username, and password in
        >3. Check database existence and user permissions
        >4. Check for firewall or network issues
        >5. Ensure psycopg2 library is installed and up to date
        >6. If using a cloud-hosted PostgreSQL, ensure that your IP address is whitelisted
        >7. If using SSL, ensure that the SSL settings in the .env file are correct
        >8. If using a connection string, ensure that it is formatted correctly and contains all necessary parameters
        >9. If using a connection pool, ensure that the pool is configured correctly and has available connections
          """)
except Exception as e:
    print(f"[ERROR] An unexpected error occurred while connecting to PostgreSQL: {e}")
    raise

[SUCCESS] Connected to PostgreSQL: PostgreSQL 18.4 on x86_64-windows, compiled by msv...



# 4. Define the tables to migrate

## Migration order

 - Categories (no dependencies)
 - Suppliers (no dependencies)
 - Customers (no dependencies)
 - Products (depends on Categories and Suppliers)

In [16]:
tables_to_migrate = ['Categories', 'Suppliers', 'Customers', 'Products']

In [17]:
print("Tables to migrate:")
for i, table in enumerate(tables_to_migrate,1):
    print(f" {i}. {table}")

total_no_tables = len(tables_to_migrate)
print(f"\nTotal no of tables to migrate: {total_no_tables}")

Tables to migrate:
 1. Categories
 2. Suppliers
 3. Customers
 4. Products

Total no of tables to migrate: 4


# 5. Run pre-migration checks

In [18]:
print("=" * 50)
print(">>> Check 1:ROW COUNTS")
print("=" * 50)

>>> Check 1:ROW COUNTS


In [19]:
test_query = "SELECT COUNT(*) as total_rows FROM Customers;"
sql_cursor.execute(test_query)
test_count = sql_cursor.fetchone()[0]
print(f"Result: {test_count}")

Result: 900000


In [20]:
baseline_count = {}

try:
    for table in tables_to_migrate:
        row_count_query = f"SELECT COUNT(*) as total_rows FROM {table}"
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_count[table] = count
        print(f"{table:10}:{count:15,} rows")

    total_rows = sum(baseline_count.values())
    print("=" * 50)
    print(f"{'TOTAL':10}:{total_rows:15,} rows\n")
except Exception as e:
    print(f"[ERROR] An error occurred while fetching row counts: {e}")
    raise

Categories:              8 rows
Suppliers :          5,000 rows
Customers :        900,000 rows
Products  :        150,000 rows
TOTAL     :      1,055,008 rows



In [ ]:
print("=" * 50)
print(">>> Check 2: NULL COUNTS (CustomerName)")
print("=" * 50)

>>> Check 2: NULL COUNTS (CustomerName)


In [ ]:
quality_issues = []

try:
    print("\n CHECK 2: NULL CHECKS (CustomerName)")
    sql_cursor.execute("""
        SELECT COUNT (*) as null_count
        FROM Customers
        WHERE CustomerName IS NULL
                       """)
    null_names = sql_cursor.fetchone()[0]
    if null_names > 0:
        quality_issues.append(f"CustomerName has {null_names:,} NULL values...")
    # print(quality_issues)

    print("\n CHECK 3: INVALID EMAIL FORMATS")
    sql_cursor.execute("""
        SELECT COUNT(*) as invalid_email_count
        FROM Customers
        WHERE Email 
        IS NOT NULL AND Email NOT LIKE '%_@__%.__%'
        OR Email Like '%@invalid'
    """)
    invalid_emails = sql_cursor.fetchone()[0]
    if invalid_emails > 0:
        quality_issues.append(f"Email has {invalid_emails:,} invalid formats...")
    # print(quality_issues)

    print("\n CHECK 4: NEGATIVE PRODUCT PRICES")
    sql_cursor.execute("""
        SELECT COUNT(*) as negative_price_count
        FROM Products
        WHERE UnitPrice < 0
    """)
    negative_prices = sql_cursor.fetchone()[0]
    if negative_prices > 0:
        quality_issues.append(f"Products have {negative_prices:,} negative prices...")
    # print(quality_issues)

    print("\n CHECK 5: NEGATIVE STOCK QUANTITIES")
    sql_cursor.execute("""
        SELECT COUNT(*) as negative_stock_count
        FROM Products
        WHERE StockQuantity < 0
    """)
    negative_stock = sql_cursor.fetchone()[0]
    if negative_stock > 0:
        quality_issues.append(f"Products have {negative_stock:,} negative stock quantities...")
    # print(quality_issues)

    print("\n CHECK 6: ORPHANED FOREIGN KEYS")
    sql_cursor.execute("""
        SELECT COUNT(*) AS orphaned_fk_count
        FROM Products p
        LEFT JOIN Suppliers s ON p.SupplierID = s.SupplierID
        WHERE s.SupplierID IS NULL
    """)
    orphaned_fks = sql_cursor.fetchone()[0]
    if orphaned_fks > 0:
        quality_issues.append(f"Products have {orphaned_fks:,} orphaned foreign keys...")
    # print(quality_issues)


    print("\n CHECK 7: FUTURES DATES CHECKS (CreatedAt)")
    sql_cursor.execute("""
        SELECT COUNT(*) as future_date_count
        FROM Customers
        WHERE CreatedDate > GETDATE()
    """)
    future_dates = sql_cursor.fetchone()[0]
    if future_dates > 0:
        quality_issues.append(f"Customers have {future_dates:,} future dates...")
    # print(quality_issues)

    if quality_issues:
        print("\nData Quality Issues Found (will migrate as-is):")
        for issue in quality_issues:
            print(f" - {issue}")
    else:
        print("\nNo data quality issues found.")

except Exception as e:
    print(f"[ERROR] ===> Unexpected error issue {e}")
    raise


 CHECK 2: NULL CHECKS (CustomerName)

 CHECK 3: INVALID EMAIL FORMATS

 CHECK 4: NEGATIVE PRODUCT PRICES

 CHECK 5: NEGATIVE STOCK QUANTITIES

 CHECK 6: ORPHANED FOREIGN KEYS

 CHECK 7: FUTURES DATES CHECKS (CreatedAt)

Data Quality Issues Found:
 - CustomerName has 4,514 NULL values...
 - Email has 8,844 invalid formats...
 - Products have 775 negative prices...
 - Products have 1,467 negative stock quantities...
 - Products have 24,700 orphaned foreign keys...
 - Customers have 8,149 future dates...


# 6. Get table schema

In [39]:
print("=" * 50)
print("ANALYZE TABLE SCHEMA")
print("=" * 50)

ANALYZE TABLE SCHEMA


In [44]:
table_schema = {}

try:
    for table in tables_to_migrate:
        schema_query=f"""
        SELECT
            COLUMN_NAME,
            DATA_TYPE,
            CHARACTER_MAXIMUM_LENGTH,
            IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = 'Customers'
        ORDER BY ORDINAL_POSITION
    """
        schema_df = pd.read_sql(schema_query, sql_conn)
        print(schema_df)
except Exception as e:
    pass

    COLUMN_NAME DATA_TYPE  CHARACTER_MAXIMUM_LENGTH IS_NULLABLE
0    CustomerID       int                       NaN          NO
1  CustomerName  nvarchar                     100.0         YES
2         Email  nvarchar                     100.0         YES
3         Phone  nvarchar                      20.0         YES
4       Country  nvarchar                     100.0         YES
5   CreatedDate  datetime                       NaN         YES
6      IsActive       bit                       NaN         YES
    COLUMN_NAME DATA_TYPE  CHARACTER_MAXIMUM_LENGTH IS_NULLABLE
0    CustomerID       int                       NaN          NO
1  CustomerName  nvarchar                     100.0         YES
2         Email  nvarchar                     100.0         YES
3         Phone  nvarchar                      20.0         YES
4       Country  nvarchar                     100.0         YES
5   CreatedDate  datetime                       NaN         YES
6      IsActive       bit               

C:\Users\user\AppData\Local\Temp\ipykernel_28260\1457657564.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)
C:\Users\user\AppData\Local\Temp\ipykernel_28260\1457657564.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)
C:\Users\user\AppData\Local\Temp\ipykernel_28260\1457657564.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  schema_df = pd.read_sql(schema_query, sql_conn)
C:\Users\user\AppData\Local\Temp\ipykernel_282